# 2D Poisson problem with TV denoising (postprocess and plots)

### Imports

In [ ]:
from cuqi.samples import Samples
from cuqipy_fenics.testproblem import FEniCSPoisson2D
import numpy as np
import cuqi
import dolfin as dl
import matplotlib.pyplot as plt

# set logging level of dl
dl.set_log_level(dl.LogLevel.ERROR)

In [ ]:
import matplotlib 

fontsize = 20
legendfontsize = 20

matplotlib.rc('xtick', labelsize=fontsize) 
matplotlib.rc('ytick', labelsize=fontsize) 

### Parameters

In [ ]:
nx = 32

In [ ]:
figure_path = "myula_tv_figs/"

### Forward model

In [ ]:
A = FEniCSPoisson2D(dim=(nx,nx), field_type=None, mapping='exponential', bc_types=['Dirichlet', 'Dirichlet', 'Dirichlet', 'Dirichlet']).model


In [ ]:
dl.plot(A.domain_geometry.mesh)

### Create true signal

In [ ]:
# set a 2D signal with a square in the middle
fun_x_true_expr = dl.Expression('(x[0]>0.35 && x[0]< 0.65 && x[1]>0.35 && x[1]< 0.65) ? 0 : -0.5', degree=1)
x_true_fun = dl.interpolate(fun_x_true_expr, A.domain_geometry.function_space)
x_true = cuqi.array.CUQIarray(x_true_fun.vector().get_local(), geometry=A.domain_geometry)

In [ ]:
fig, ax = plt.subplots()
im = x_true.plot(subplots=False, vmin=-0.5, vmax=0.0, mode='color')
inset_axes = ax.inset_axes([1.04, 0.2, 0.05, 0.6])
fig.colorbar(im[0], ax=ax, cax=inset_axes)
ax.set_ylim(0, 1)
ax.set_xlim(0, 1)
ax.set_xlabel('')
ax.set_ylabel('')
plt.savefig(figure_path + "x_true.pdf", bbox_inches='tight')

### Create true data

In [ ]:

y_true = A(x_true)

In [ ]:
fig, ax = plt.subplots()
im = y_true.plot(subplots=False, vmin=0, vmax=0.12, mode='color')
inset_axes = ax.inset_axes([1.04, 0.2, 0.05, 0.6])
fig.colorbar(im[0], ax=ax, cax=inset_axes)
ax.set_ylim(0, 1)
ax.set_xlim(0, 1)
ax.set_xlabel('')
ax.set_ylabel('')
plt.savefig(figure_path + "y_true.pdf", bbox_inches='tight')

### Plot results

In [ ]:
# Read all the cases for reg strength 
dir = "results/"
file_list = [
"posterior_samples_2D_nx_32_rest_str_1.0_Ns_500000.npz",
"posterior_samples_2D_nx_32_rest_str_5.0_Ns_500000.npz",
"posterior_samples_2D_nx_32_rest_str_7.0_Ns_500000.npz",
"posterior_samples_2D_nx_32_rest_str_10.0_Ns_500000.npz",
"posterior_samples_2D_nx_32_rest_str_20.0_Ns_500000.npz",
"posterior_samples_2D_nx_32_rest_str_30.0_Ns_500000.npz"]

rest_str_factor = [1.0, 5.0, 7.0, 10.0, 20.0, 30.0]

data_list = []
y_obs_list = []

for file in file_list:
    data = np.load(dir+file)
    data_list.append(Samples(data['posterior_samples'], geometry=A.domain_geometry))
    y_obs_list.append(data['noisy_data'])

In [ ]:
for i in range(len(rest_str_factor)):
    fig, ax = plt.subplots()
    mean_i = cuqi.array.CUQIarray(data_list[i].mean(), geometry=A.domain_geometry)
    im = mean_i.plot(subplots=False, vmin=-0.5, vmax=0.0, mode='color')
    inset_axes = ax.inset_axes([1.04, 0.2, 0.05, 0.6])
    fig.colorbar(im[0], ax=ax, cax=inset_axes)
    ax.set_ylim(0, 1)
    ax.set_xlim(0, 1)
    ax.set_xlabel('')
    ax.set_ylabel('')
    plt.savefig(figure_path + f"mean_{int(rest_str_factor[i])}.pdf", bbox_inches='tight')

### Plot case: regularization strength = 7

In [ ]:
expr_idx = 2

Plot the noisy data

In [ ]:
fig, ax = plt.subplots()
y_obs_cuqi_array = cuqi.array.CUQIarray(y_obs_list[expr_idx], geometry=A.range_geometry)
im = y_obs_cuqi_array.plot(subplots=False, vmin=0, vmax=0.12, mode='color')
inset_axes = ax.inset_axes([1.04, 0.2, 0.05, 0.6])
fig.colorbar(im[0], ax=ax, cax=inset_axes)
ax.set_ylim(0, 1)
ax.set_xlim(0, 1)
ax.set_xlabel('')
ax.set_ylabel('')
plt.savefig(figure_path + "y_obs.pdf", bbox_inches='tight')

In [ ]:
fig, ax = plt.subplots()
this_mean = cuqi.array.CUQIarray(data_list[expr_idx].std(), geometry=A.domain_geometry)
im = this_mean.plot(subplots=False, vmin=0.08, vmax=0.24, mode='color')
inset_axes = ax.inset_axes([1.04, 0.2, 0.05, 0.6])
# fig.colorbar(im[0], ax=ax, cax=inset_axes)
cbar = plt.colorbar(im[0], ax=ax, cax=inset_axes)
cbar.set_ticks([0.08, 0.16, 0.24])
cbar.set_ticklabels(['0.08 ', '0.16 ', '0.24 '])
ax.set_ylim(0, 1)
ax.set_xlim(0, 1)
ax.set_xlabel('')
ax.set_ylabel('')
plt.savefig(figure_path + "std.pdf", bbox_inches='tight')

In [ ]:
xi_1 = np.linspace(0, 1, 100)
exact_line = []
for xi_1_i in xi_1:
        exact_line.append(x_true_fun(xi_1_i, 0.5))
exact_line = np.array(exact_line)
line_samples = []
xi_1 = np.linspace(0, 1, 100)
samples_funvals = data_list[expr_idx].funvals
for j, fun_i in enumerate(samples_funvals):
    temp_list = []
    for xi_1_i in xi_1:
        temp_list.append(fun_i(xi_1_i, 0.5))
    line_samples.append(temp_list)
line_samples = np.array(line_samples)
line_samples_obj = cuqi.samples.Samples(line_samples.T, geometry=cuqi.geometry.Continuous1D(xi_1))

In [ ]:
plt.figure()
lines = line_samples_obj.plot_ci(exact=exact_line)
plt.legend(["95% CI", "Mean", "Exact"], fontsize=18, loc="upper left", ncol=1, frameon=False)
plt.xlim([0,1])
plt.ylim([-1, 0.5])
plt.gca().set_box_aspect(1)
plt.savefig(figure_path + "line_ci.pdf", bbox_inches='tight')